# Resume occlusion specialist on Kaggle
Create a PRIVATE Kaggle dataset containing the complete existing occlusion folder (last.pt, config.yaml, data_signature.json, history and logs), then attach it to this notebook. Do not upload credentials. Select GPU, enable Internet for setup, and run cells in order. Do not use Run All before setting the checkpoint path. Existing training configuration and fingerprinted code must match. Only one GPU is exposed because the Colab checkpoint contains one CUDA RNG state.

Kaggle input datasets are read-only. This notebook copies the run into /kaggle/working/task2-specialists/occlusion before resuming. Working files are not a substitute for a saved notebook output version or downloaded backup. Save outputs after training and before ending the session. No Google Drive mount is used.


In [ ]:
# Run before importing torch; restart the session first if torch already initialized CUDA.
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '0'
from pathlib import Path
import subprocess, sys
repo = Path('/kaggle/working/GenAI_A1')
if not (repo / '.git').exists():
    subprocess.run(['git', 'clone', 'https://github.com/AaizUrRehmanKazmi/GenAI_A1.git', str(repo)], check=True)
os.chdir(repo)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', 'requirements-training.txt'], check=True)


If installation requests a restart, restart and rerun the setup cell before continuing. This does not reinstall PyTorch; use the GPU-enabled version provided by Kaggle.


In [ ]:
import torch
print('PyTorch:', torch.__version__)
assert torch.cuda.is_available(), 'Select a GPU accelerator in Kaggle settings'
assert torch.cuda.device_count() == 1, 'Restart and run the CUDA visibility cell first'
print('GPU:', torch.cuda.get_device_name(0))
print('Attached checkpoints:')
for p in Path('/kaggle/input').rglob('last.pt'):
    print(p)


In [ ]:
# Replace with the OCCLUSION last.pt path printed above.
source_checkpoint = Path('/kaggle/input/REPLACE_WITH_YOUR_DATASET/occlusion/last.pt')
assert source_checkpoint.is_file(), 'Attach the occlusion dataset and set its exact checkpoint path'
saved = torch.load(source_checkpoint, map_location='cpu', weights_only=True)
assert saved['config'].get('condition') == 'occlusion', 'Wrong specialist checkpoint'
assert saved['config'].get('architecture') == 'specialist16_ae_v1', 'Wrong architecture'
assert not saved['data_signature']['train_limit'] and not saved['data_signature']['val_images'], 'This is a debug checkpoint'
print('Completed epochs:', saved['progress']['epoch'], 'Batch cursor:', saved['progress']['cursor'])
print('Configured target:', saved['config']['training']['epochs'])
assert saved['progress']['epoch'] < saved['config']['training']['epochs'], 'Already complete; do not retrain'


In [ ]:
# Fail before dataset download if the repository does not match checkpoint provenance.
import hashlib
for relative, expected in saved['data_signature']['source'].items():
    path = repo / relative
    assert path.is_file() and hashlib.sha256(path.read_bytes()).hexdigest() == expected, (
        f'Source mismatch: {relative}. Restore the Git revision used for this run; do not bypass this check.'
    )


In [ ]:
raw = repo / 'data/raw/oxford_pets'
raw.mkdir(parents=True, exist_ok=True)
for name in ['images', 'annotations']:
    archive = raw / f'{name}.tar.gz'
    subprocess.run(['wget', '-c', '-P', str(raw), f'https://thor.robots.ox.ac.uk/pets/{name}.tar.gz'], check=True)
    subprocess.run(['tar', '-xzf', str(archive), '-C', str(raw)], check=True)
subprocess.run([sys.executable, '-m', 'scripts.prepare_pets'], check=True)
subprocess.run([sys.executable, '-m', 'scripts.generate_manifests'], check=True)


In [ ]:
import shutil, yaml
output = Path('/kaggle/working/task2-specialists/occlusion')
if not output.exists():
    shutil.copytree(source_checkpoint.parent, output)
assert (output / 'last.pt').is_file(), 'Existing output is incomplete; preserve it and inspect before continuing'
local = torch.load(output / 'last.pt', map_location='cpu', weights_only=True)
assert local['config'] == saved['config'], 'Existing output belongs to another run'
# Trainer adds the condition-specific seed offset. Recover the original base seed.
config = dict(local['config'])
config['seed'] = config['seed'] - 303
config.pop('condition')
config_path = Path('/kaggle/working/occlusion_resume.yaml')
config_path.write_text(yaml.safe_dump(config))
print('Resuming from:', output / 'last.pt')


In [ ]:
# Soft budget leaves time to save/download outputs. Adjust downward if quota is low.
subprocess.run([sys.executable, '-m', 'training.train_specialists',
    '--condition', 'occlusion', '--config', str(config_path),
    '--device', 'cuda', '--max-hours', '2',
    '--output-dir', str(output), '--resume', str(output / 'last.pt')], cwd=repo, check=True)


In [ ]:
import json
from IPython.display import display, Image, FileLink
history = json.loads((output / 'history.json').read_text())
if history:
    best = min(history, key=lambda row: row['val_loss'])
    print('Completed:', history[-1]['epoch'], 'Best epoch:', best['epoch'])
    print('Loss:', best['val_loss'], 'L1:', best['val_l1'], 'SSIM:', best['val_ssim'])
    if (output / 'preview.png').exists():
        display(Image(filename=str(output / 'preview.png')))
else:
    print('No complete epoch yet; saved cursor can be resumed')
backup = shutil.make_archive('/kaggle/working/occlusion_backup', 'zip', output.parent, output.name)
display(FileLink(backup))


## Preserve results
Download occlusion_backup.zip, or save a notebook version that includes these output files and verify that they appear in the saved version. Do not rely on temporary session storage. For another session, attach the NEW backup as a private dataset and point source_checkpoint to its last.pt. Keep the original Colab backup untouched.

A changed GPU/PyTorch environment may change numerical results; environment.json records the current runtime. The trainer enforces config/data/source matching. If it stops below20 epochs, resume from the newest saved checkpoint. The trained salt/blur models and classifier do not need moving just to train occlusion. Share history.json and preview.png when done.
